This is the group project submission for Merina, Gabriel, and Dominic.

We chose to work on the Snowflake Fingerprintability project on the nPrint PCAPML leaderboard. It was aimed to differentiate Snowflake Tor DTLS handshakes from other WebRTS traffic. DTLS (Datagram Transport Layer Security) is one of the main forms of security protocols used by WebRTC traffic. There are 7 different types of traffic analyzed in the dataset we are using: Snowflake, Facebook Messenger, Discord, and Google hangouts, across Google Chrome and Firefox browsers. According to the paper cited in the study.  

There are 2 main, important differences between the Snowflake traffic and the other types of connections. The first, the Snowflake handshakes contain significantly more packets: 13.2 to around 5 for the other types of traffic(Macmillan et al.). The other is that there are 2 features that are present in Snowflake traffic that is missing from all the other traffic: "Server Message Sequence: '1'" and "supported_groups." There is also a feature, "renegotiation info," that is missing from Snowflake traffic, but included in the other types of traffic. A third, now-spurious correlation is that all Snowflake traffic occured on Firefox: The extension is now available on Chrome as well, so this is no longer a viable way to differentiate this traffic.  

To differentiate the traffic, we will:

- Label all of the packet capture data from the given handshake sets, so we can create training data from it
- Train the Model on the labeled handshake data from the paper's github repo
- Load the packet capture of the case study as test data
- Test the model on the data
- Visualize the results (Accuracy, AUC Graph, Confusion Matrix)

In [4]:
import glob
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from netml.pparser.parser import PCAP
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import GroupShuffleSplit

In [5]:
DATA_DIR = "webrtc-handshakes"
CATEGORIES = ["discord", "facebook", "google", "snowflake"]
FEAT_TYPE = "STATS"
FEATURE_COLS = [
    "duration", "pkts_rate", "bytes_rate", "size_mean", "size_std",
    "size_q1", "size_q2", "size_q3", "size_min", "size_max",
    "num_pkts", "num_bytes",
]
#Initializing constants for the model to extract useful feautres
# and be able to classify based on the categories avaialable

In [6]:
def pcap_to_flow_features(pcap_file, feat_type=FEAT_TYPE, flow_ptks_thres=2):
    """Parse one pcap file into netml flows and return (features, fids)."""
    pp = PCAP(pcap_file, flow_ptks_thres=flow_ptks_thres, verbose=0)
    pp.pcap2flows()
    if len(pp.flows) == 0:
        return np.empty((0, len(FEATURE_COLS))), []
    pp.flow2features(feat_type=feat_type)
    return pp.features, pp.fids

In [7]:
"""
This block will take all the pcap files in the directory, extract the flows,
and extract features from the flows. It will create a dataframe from a list of
lists containing specific features and the labels of the data. This will help us 
create training data for our model to classify the data based on the features 
extracted from the flows.
"""

records = []
for category in CATEGORIES:
    pcap_files = glob.glob(os.path.join(DATA_DIR, category, "*.pcap"))
    for pcap_file in pcap_files:
        features, fids = pcap_to_flow_features(pcap_file)
        for feat, fid in zip(features, fids):
            src_ip, dst_ip, src_port, dst_port, protocol = fid
            records.append({
                "file": pcap_file,
                "label": category,
                "src_ip": src_ip,
                "dst_ip": dst_ip,
                "src_port": src_port,
                "dst_port": dst_port,
                "protocol": protocol,
                **dict(zip(FEATURE_COLS, feat)),
            })

flows_df = pd.DataFrame.from_records(records)
print(flows_df.shape)
print(flows_df["label"].value_counts())

(8773, 19)
label
discord      3346
snowflake    1932
facebook     1824
google       1671
Name: count, dtype: int64
